# TTC Multi-Mode Delay Analysis

This notebook loads TTC delay datasets from `../data/bus`, `../data/subway`, and `../data/streetcar`, then builds yearly, monthly, weekly, and incident/category summaries.


In [ ]:
from pathlib import Path
import re

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path('../data')
MODE_FOLDERS = ('bus', 'subway', 'streetcar')
FILE_PATTERN = '*.csv'
EXCLUDED_TOKENS = ('readme', 'code description', 'code-descriptions')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid')


In [ ]:
COLUMN_ALIASES = {
    'date': 'date',
    'report_date': 'date',
    'incident_date': 'date',
    'time': 'time',
    'report_time': 'time',
    'incident_time': 'time',
    'time_of_day': 'time',
    'route': 'route',
    'route_number': 'route',
    'route_name': 'route',
    'route_no': 'route',
    'route_num': 'route',
    'line': 'route',
    'station': 'location',
    'location': 'location',
    'location_name': 'location',
    'intersection': 'location',
    'incident': 'incident',
    'incident_type': 'incident',
    'incident_description': 'incident',
    'delay_reason': 'incident',
    'code': 'incident_code',
    'min_delay': 'min_delay',
    'min delay': 'min_delay',
    'delay_minutes': 'min_delay',
    'delay': 'min_delay',
    'min_gap': 'min_gap',
    'min gap': 'min_gap',
    'gap_minutes': 'min_gap',
    'gap': 'min_gap',
    'day': 'day',
    'direction': 'direction',
    'bound': 'direction',
    'vehicle': 'vehicle',
}

REQUIRED_COLUMNS = ['date', 'route', 'location', 'min_delay', 'min_gap']

def normalize_col(col: str) -> str:
    col = str(col).strip().strip('\"').replace('﻿', '').replace('​', '')
    col = col.replace('/', '_').replace('-', '_').lower()
    col = re.sub(r'[^a-z0-9_]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return COLUMN_ALIASES.get(col, col)

def read_any_table(file_path: Path) -> pd.DataFrame:
    payload = file_path.read_bytes()
    is_excel_payload = payload[:2] == b'PK'

    if file_path.suffix.lower() in {'.xlsx', '.xls'} or is_excel_payload:
        return pd.read_excel(file_path)

    encodings = ('utf-8', 'utf-8-sig', 'utf-16', 'cp1252', 'latin1')
    separators = (None, ',', '\t', ';', '|')

    for encoding in encodings:
        for separator in separators:
            try:
                return pd.read_csv(file_path, encoding=encoding, sep=separator, engine='python', on_bad_lines='skip')
            except Exception:
                pass

    raise ValueError(f'Unable to parse file: {file_path}')

def categorize_incident(text: str) -> str:
    value = str(text).strip().lower()
    if not value or value == 'nan':
        return 'Unknown'

    keyword_map = {
        'Mechanical': ['mechanical', 'engine', 'brake', 'vehicle', 'equipment'],
        'Traffic': ['traffic', 'congestion', 'slow traffic'],
        'Security': ['security', 'police', 'investigation'],
        'Diversion': ['diversion', 'detour', 'road closure'],
        'Collision': ['collision', 'accident', 'crash'],
        'Medical': ['medical', 'sick customer', 'injury'],
        'Weather': ['weather', 'snow', 'storm', 'rain'],
        'Passenger': ['passenger', 'customer', 'wheelchair'],
        'Operations': ['operator', 'crew', 'dispatch', 'late leaving', 'general delay'],
        'Construction': ['construction', 'work zone'],
    }

    for label, keywords in keyword_map.items():
        if any(keyword in value for keyword in keywords):
            return label

    if re.fullmatch(r'[A-Z]{3,6}', str(text).strip()):
        return f'Code-{str(text).strip()[:2]}'

    return str(text).strip().title()

def standardize_dataset(df: pd.DataFrame, file_path: Path, mode: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_col(column) for column in df.columns]

    for target, patterns in {
        'date': ('date',),
        'route': ('route', 'line'),
        'location': ('location', 'station', 'intersection', 'stop'),
        'incident': ('incident',),
        'incident_code': ('code',),
        'min_delay': ('delay',),
        'min_gap': ('gap',),
    }.items():
        if target in df.columns:
            continue
        match = next((col for col in df.columns if any(pattern in col for pattern in patterns)), None)
        if match is not None:
            df[target] = df[match]

    if 'min_gap' not in df.columns and 'min_delay' in df.columns:
        df['min_gap'] = df['min_delay']

    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f'{file_path.name} is missing required columns: {missing}')

    if 'incident' not in df.columns and 'incident_code' in df.columns:
        df['incident'] = df['incident_code']

    if 'incident_code' not in df.columns:
        df['incident_code'] = pd.NA

    standardized = pd.DataFrame()
    standardized['date'] = pd.to_datetime(df['date'], errors='coerce')
    standardized['route'] = df['route'].astype(str).str.strip()
    standardized['location'] = df['location'].astype(str).str.strip()
    standardized['incident'] = df['incident'].astype(str).str.strip()
    standardized['incident_category'] = standardized['incident'].apply(categorize_incident)
    standardized['min_delay'] = pd.to_numeric(df['min_delay'], errors='coerce').fillna(0)
    standardized['min_gap'] = pd.to_numeric(df['min_gap'], errors='coerce').fillna(0)
    standardized['mode'] = mode
    standardized['source_file'] = file_path.name

    standardized = standardized.dropna(subset=['date'])
    standardized['year'] = standardized['date'].dt.year
    standardized['month'] = standardized['date'].dt.month
    standardized['month_name'] = standardized['date'].dt.month_name().str.slice(0, 3)
    standardized['week'] = standardized['date'].dt.isocalendar().week.astype('int64')
    standardized['day_name'] = standardized['date'].dt.day_name()

    return standardized

def load_all_datasets(data_dir: Path):
    files = sorted(
        path
        for mode in MODE_FOLDERS
        for path in (data_dir / mode).glob(FILE_PATTERN)
        if not any(token in path.name.lower() for token in EXCLUDED_TOKENS)
    )

    frames = []
    file_summary_rows = []
    failures = []

    for file_path in files:
        mode = file_path.parent.name.lower()
        try:
            raw = read_any_table(file_path)
            clean = standardize_dataset(raw, file_path, mode)
            frames.append(clean)
            file_summary_rows.append({
                'mode': mode,
                'file_name': file_path.name,
                'rows_loaded': len(clean),
                'min_year': clean['year'].min(),
                'max_year': clean['year'].max(),
            })
        except Exception as exc:
            failures.append((mode, file_path.name, str(exc)))

    if not frames:
        raise ValueError('No usable dataset files were loaded from the data directory.')

    combined = pd.concat(frames, ignore_index=True)
    file_summary = pd.DataFrame(file_summary_rows).sort_values(['mode', 'min_year', 'file_name']).reset_index(drop=True)
    return combined, file_summary, failures


In [ ]:
df, file_summary, failures = load_all_datasets(DATA_DIR)

print(f'Total rows loaded: {len(df):,}')
print(f'Years covered: {int(df["year"].min())} to {int(df["year"].max())}')
print(f'Modes: {sorted(df["mode"].unique().tolist())}')
print(f'Files loaded: {len(file_summary):,}')
print(f'Files skipped due to parsing/shape issues: {len(failures):,}')

display(file_summary.head(30))
if failures:
    display(pd.DataFrame(failures, columns=['mode', 'file_name', 'error']))


In [ ]:
yearly = (
    df.groupby(['mode', 'year'], as_index=False)
      .agg(events=('year', 'size'), total_delay=('min_delay', 'sum'), total_gap=('min_gap', 'sum'))
      .sort_values(['mode', 'year'])
)

display(yearly.head(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=yearly, x='year', y='events', hue='mode', marker='o', ax=axes[0])
axes[0].set_title('Incidents by Year')

sns.lineplot(data=yearly, x='year', y='total_delay', hue='mode', marker='o', ax=axes[1])
axes[1].set_title('Total Delay Minutes by Year')

plt.tight_layout()
plt.show()


In [ ]:
monthly = (
    df.groupby(['mode', 'year', 'month'], as_index=False)
      .agg(events=('month', 'size'), total_delay=('min_delay', 'sum'))
      .sort_values(['mode', 'year', 'month'])
)
weekly = (
    df.groupby(['mode', 'year', 'week'], as_index=False)
      .agg(events=('week', 'size'), total_delay=('min_delay', 'sum'))
      .sort_values(['mode', 'year', 'week'])
)

display(monthly.head(20))
display(weekly.head(20))


In [ ]:
incident_categories = (
    df.groupby(['mode', 'incident_category'], as_index=False)
      .agg(events=('incident_category', 'size'), total_delay=('min_delay', 'sum'))
      .sort_values(['mode', 'events'], ascending=[True, False])
)

top_routes = (
    df.groupby(['mode', 'route'], as_index=False)
      .agg(events=('route', 'size'), total_delay=('min_delay', 'sum'), total_gap=('min_gap', 'sum'))
      .sort_values(['mode', 'events'], ascending=[True, False])
)

display(incident_categories.head(30))
display(top_routes.head(30))
